# Exploratory data analysis

Run from top to bottom. Quick mode is a patient subset, not final results. Notebook outputs are separate from production reports.

In [ ]:
from pathlib import Path
import sys, os
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p/'src'/'features.py').exists()), None)
assert ROOT is not None, 'Open the notebook from the project repository'
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from src.notebook_utils import *
QUICK = True  # False uses the entire eligible dataset; keep consistent across notebooks
OUT = ROOT/'reports'/('notebook_quick' if QUICK else 'notebook_full')
OUT.mkdir(parents=True, exist_ok=True)


Patient-disjoint development cohort

In [ ]:
raw, x, y, g, parts = load_cohort(ROOT, QUICK)
train, cal, tune, test = (parts[k] for k in ("train","calibration","threshold","test"))
X, Y, G = x.iloc[train], y.iloc[train], g.iloc[train]
display(split_table(y,g,parts))

Outcome distribution and numerical distributions

In [ ]:
Y.value_counts().sort_index().plot.bar(title='Training outcome counts'); plt.show()
X[['time_in_hospital','num_medications','number_inpatient','num_lab_procedures']].hist(figsize=(10,6)); plt.tight_layout(); plt.show()

Readmission rate by age and previous inpatient visits

In [ ]:
for c in ['age','number_inpatient']:
    z=pd.DataFrame({'feature':X[c],'target':Y}).groupby('feature').target.agg(['size','mean']); display(z)
    z['mean'].plot.bar(figsize=(10,3),title=f'Observed readmission rate by {c}'); plt.ylabel('Fraction'); plt.show()

Numerical association heatmap and statistical tests. Encounter-level p-values are descriptive because patients can repeat.

In [ ]:
from src.eda import run
run(X,Y,OUT/'eda'); a=pd.read_csv(OUT/'eda/associations.csv'); display(a.sort_values('p_bonferroni').head(15))
corr=X.select_dtypes('number').corr(method='spearman'); fig,ax=plt.subplots(figsize=(8,6)); im=ax.imshow(corr,vmin=-1,vmax=1,cmap='coolwarm'); ax.set_xticks(range(len(corr)),corr.columns,rotation=90); ax.set_yticks(range(len(corr)),corr.columns); fig.colorbar(im); plt.tight_layout(); plt.show()